[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/ruppeiner/ruppeiner.ipynb)

# Fluctuations Near the Critical Point: the Ruppeiner Metric

This example computes the fluctuation metric of a van der Waals fluid, the cost of a small variation of its temperature and volume away from a stable state: the larger the cost, the less likely the fluid fluctuates that far. Near the critical point the fluid's pressure hardly changes with its volume, so variations of the volume cost almost nothing and the fluid fluctuates wildly, which shows as the ellipses of equal cost stretching along the volume and as the metric's curvature growing without bound.

A state is modelled as an antivector of the plane spanned by temperature, `mv.t.dual()`, and volume, `mv.v.dual()`. The metric is a map from a variation of the state to a covector, `Covector <- State`, which pairs with the variation to its cost, `metric(state)(variation) & variation`: the reverse of a mobility, which maps a covector to a rate. It is built from the Helmholtz energy's second derivative in its own variables, temperature and volume, the thermal part with its sign turned and both over the temperature; in these variables it has no cross term. The ellipses of equal cost are the level sets of that pairing, drawn as conics.

In the usual notation the metric is $g = \frac{C_V}{T^2}\,dT^2 - \frac{1}{T}\left(\frac{\partial p}{\partial V}\right)_T dV^2$, and the probability of a fluctuation falls as $e^{-g/2}$. Its scalar curvature, here $R = 2K$, positive on a sphere (Ruppeiner's sign convention is the opposite), grows as $(T - T_c)^{-2}$ toward the critical point along the critical volume.

Temperature and volume are in their critical values, pressure in $RT_c/V_c$, energy in $RT_c$; the gas constant is one, so the critical pressure is $3/8$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra
from numga.backend.jax import JaxContext, derivative
from examples.util.animation import show_animation
from examples.thermodynamics.ruppeiner import render

ga = Algebra("t0v0")
mv = JaxContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()
State = ga.gatype.antivector()
Covector = ga.gatype.vector()
Metric = ga.gatype((Covector, State))

# The two readings a state is described by.
temperature = mv.t & State                                    # Scalar <- State
volume = mv.v & State                                         # Scalar <- State
# The fluid: the strength of its molecules' attraction, the volume they exclude, and its heat capacity at fixed volume.
attraction, exclusion, capacity = 9 / 8, 1 / 3, 1.5

## The fluctuation metric

The Helmholtz energy of the fluid at a temperature and volume is differentiated twice; the second derivative is a form on pairs of variations. Bound to a variation of temperature twice it gives the heat capacity over the temperature with its sign turned, bound to a variation of volume twice minus the pressure's slope along the volume. The metric puts the two together as a map from a variation to a covector, `mv.t * thermal * temperature + mv.v * mechanical * volume`. At a state well above the critical point both entries are positive.

In [ ]:
def pressure(state: State) -> Scalar:
    """The van der Waals equation of state."""
    return temperature(state) / (volume(state) - exclusion) - attraction / volume(state).squared()

def helmholtz_energy(state: State) -> Scalar:
    warmth, room = temperature(state), volume(state)
    return -capacity * warmth * warmth.log() - warmth * (room - exclusion).log() - attraction / room

def metric(state: State) -> Metric:
    """The cost of variations about a state, as a map from a variation to a covector."""
    hessian = derivative(derivative(helmholtz_energy))(state)                 # [...] Scalar <- (State, State)
    # The thermal part with its sign turned; in temperature and volume there is no cross term.
    thermal = -hessian(mv.t.dual(), mv.t.dual()) / temperature(state)
    mechanical = hessian(mv.v.dual(), mv.v.dual()) / temperature(state)
    return mv.t * thermal * temperature + mv.v * mechanical * volume

warm = mv.t.dual() * 1.8 + mv.v.dual()                                       # [] State

In [ ]:
print("cost of a unit variation of temperature:", (metric(warm)(mv.t.dual()) & mv.t.dual()).to_array())
print("cost of a unit variation of volume:     ", (metric(warm)(mv.v.dual()) & mv.v.dual()).to_array())

## The metric's curvature

How the metric changes from state to state is its curvature. For a metric with no cross term whose thermal entry depends only on the temperature, the scalar curvature needs only the mechanical entry's first and second derivatives along the temperature and the first derivative of the two entries' product. The chart shows the plane of temperature and volume: shaded below the black spinodal, where the mechanical entry turns negative and the uniform fluid is unstable; above it, teal ellipses of equal cost about a lattice of states, and the grey-blue level lines of the curvature, crowding toward the critical point at the spinodal's peak. The state above is orange.

In [ ]:
def curvature(state: State) -> Scalar:
    """The metric's scalar curvature, positive for a sphere."""
    thermal = lambda point: metric(point)(mv.t.dual()) & mv.t.dual()
    mechanical = lambda point: metric(point)(mv.v.dual()) & mv.v.dual()
    determinant = lambda point: thermal(point) * mechanical(point)
    first = derivative(mechanical)(state)(mv.t.dual())
    second = derivative(derivative(mechanical))(state)(mv.t.dual(), mv.t.dual())
    area_squared = determinant(state)
    area_change = derivative(determinant)(state)(mv.t.dual())
    return -second / area_squared + first * area_change / (2 * area_squared.squared())

render.draw_landscape(metric, curvature, pressure, warm)

## Toward the critical point

The state cools along the critical volume from 1.8 to 1.02 times the critical temperature. Its ellipse of equal cost widens along the volume while hardly changing along the temperature, and on the right its isotherm's slope at the state, the mechanical entry of the metric times the temperature, flattens.

In [ ]:
def approach():
    for warmth in np.linspace(1.8, 1.02, 48):
        yield mv.t.dual() * warmth + mv.v.dual()

show_animation(render.fluctuations(metric, curvature, pressure, approach()), "ruppeiner_fluctuations", duration_ms=80)

Along the critical volume the curvature grows as the inverse square of the distance above the critical temperature: on logarithmic scales its line runs parallel to the dashed one of slope $-2$.

In [ ]:
offsets = np.geomspace(0.002, 0.8, 100)
render.scaling(mv.t.dual() * (1 + offsets) + mv.v.dual(), curvature)

In [ ]:
# checks
states = mv.t.dual() * np.array([1.02, 1.2, 1.8]) + mv.v.dual() * np.array([1, 0.8, 1.5])
thermal = metric(states)(mv.t.dual()) & mv.t.dual()
np.testing.assert_allclose(thermal.to_array(), capacity / np.array([1.02, 1.2, 1.8]) ** 2, atol=1e-10)
np.testing.assert_allclose((metric(states)(mv.t.dual()) & mv.v.dual()).to_array(), 0, atol=1e-10)
# Close to the critical point the curvature is a third of the inverse square of the distance above it.
near = curvature(mv.t.dual() * 1.002 + mv.v.dual())
assert abs(float(near.to_array()) * 3 * 0.002 ** 2 - 1) < 0.01